In [ ]:
import time
import sys
import enum
import pandas as pd
from nba_api.stats.static import players
from sqlalchemy import update
from sqlalchemy.orm import Session

sys.path.insert(0, "..")

import models
from database import engine, sessionDB
from setup_functions.stats import getSeasonStats

In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [3]:
models.Base.metadata.create_all(bind=engine)
db = sessionDB()

active_players = players.get_active_players()
season = "2025-26"

In [10]:
pid = 203471
stats = getSeasonStats(pid)
reg, post = stats[0], stats[1]

reg = reg[reg["SEASON"] == season]
post = post[post["SEASON"] == season]

reg.info

<bound method DataFrame.info of      SEASON     TEAM_ID TEAM_ABBREVIATION PLAYER_AGE  GP  GS   MIN  FGM  FGA  \
19  2025-26  1610612758               SAC       32.0  40  14  1056  172  422   
20  2025-26  1610612739               CLE       32.0  30   3   641   83  207   
21  2025-26           0               TOT       32.0  70  17  1696  255  629   

   FG_PCT FG3M FG3A FG3_PCT  FTM  FTA FT_PCT OREB DREB  REB  AST STL BLK  TOV  \
19  0.408   57  166   0.343  109  133   0.82   22  100  122  211  32   6   79   
20  0.401   18   62    0.29   62   72  0.861   11   57   68  130  23   5   45   
21  0.405   75  228   0.329  171  205  0.834   33  157  190  341  55  11  124   

     PF  PTS  
19   66  510  
20   43  246  
21  109  756  >

In [ ]:
def upsertStats(
    db: Session,
    pid: int,
    season_type: enum.Enum,
    df: pd.DataFrame,
    season: str
) -> None:
    
